# 因果叙事分析（中文示例）

本笔记本演示如何使用 causal-narrative 包分析中文因果叙事。

In [1]:
import warnings
import sys
from pathlib import Path

# 添加本地包路径，确保导入本地开发版本
project_root = Path.cwd().parent  # 从 notebook/ 目录向上一级到项目根目录
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"Added local package path: {project_root}")

from loguru import logger

warnings.filterwarnings('ignore')
logger.remove()
logger.add(sys.stderr, level='INFO')

# 验证导入的是本地版本
import causal_narrative
print(f"Using causal_narrative from: {causal_narrative.__file__}")
print(f"Version: {causal_narrative.__version__}")

Added local package path: /Volumes/Yangdong/causal-narrative
Using causal_narrative from: /Volumes/Yangdong/causal-narrative/causal_narrative/__init__.py
Version: 0.2.0


## 1. 准备中文句子

我们使用一些中文因果句子作为示例：

In [2]:
sentences = [
    "央行提高了利率，导致物价下降。",
    "中央银行上调了基准利率，导致经济严重通缩。",
    "企业大幅裁员，导致失业率攀升",
    "央行提高了利率，导致金融系统崩溃。",
    "中央银行上调了基准利率，导致经济严重萎缩。",
    "贸易战加剧，导致出口锐减。",
    "政府削减了公共支出，导致经济衰退。",
    "政府减少了预算，导致经济增长停滞。",
    "国家大幅削减了资金，导致经济陷入低迷。",
    "疫情爆发，导致经济停滞。",
    "政府减少了预算，导致公共服务瘫痪。",
    "国家大幅削减了资金，导致医疗系统崩溃。",
    "主席昨天发表了关于国家团结的讲话。",
    "大选定于明年十一月举行。"
]

## 2. 因果关系检测

使用专门训练的**中文 BERT 模型**进行因果关系检测：
- 模型：`causal-narrative/zh-roberta-causal-narrative-classifier`
- 基于中文 RoBERTa，在中文因果语料上微调

In [3]:
from causal_narrative.detection import CausalDetector

# 使用中文 BERT 模型进行因果检测
logger.info("初始化因果检测器（中文 BERT 模型）...")
detector = CausalDetector(
    method='bert',
    bert_model_name='causal-narrative/zh-roberta-causal-narrative-classifier'  # 使用中文模型
)

logger.info("运行因果检测...")
detection_results = detector.detect(sentences, show_progress=True)

causal_sentences = []
for i, res in enumerate(detection_results):
    if res.has_causality:
        print(f"句子 {i+1} 是因果句（置信度: {res.score:.2f}）")
        causal_sentences.append(sentences[i])
    else:
        print(f"句子 {i+1} 不是因果句（置信度: {res.score:.2f}）")

print(f"\n共检测到 {len(causal_sentences)} 个因果句")

2026-03-25 17:13:59.277 | INFO     | __main__:<module>:4 - 初始化因果检测器（中文 BERT 模型）...
2026-03-25 17:13:59.278 | INFO     | causal_narrative.detection:_resolve_bert_model_source:299 - Loading BERT model from local cache: model/causal-narrative_zh-roberta-causal-narrative-classifier
2026-03-25 17:13:59.382 | INFO     | causal_narrative.detection:__init__:394 - BERTCausalDetector loaded from: model/causal-narrative_zh-roberta-causal-narrative-classifier
2026-03-25 17:13:59.382 | INFO     | causal_narrative.detection:__init__:395 - Device: cpu
2026-03-25 17:13:59.382 | INFO     | causal_narrative.detection:__init__:678 - CausalDetector initialized with BERT method (model=causal-narrative/zh-roberta-causal-narrative-classifier, cache_dir=model)
2026-03-25 17:13:59.382 | INFO     | __main__:<module>:10 - 运行因果检测...
2026-03-25 17:13:59.383 | INFO     | causal_narrative.detection:_detect_bert_batch:883 - Starting batch BERT detection for 14 sentences
BERT detection: 100%|██████████| 14/14 [00:00<0

句子 1 是因果句（置信度: 0.99）
句子 2 是因果句（置信度: 0.99）
句子 3 是因果句（置信度: 0.99）
句子 4 是因果句（置信度: 0.99）
句子 5 是因果句（置信度: 0.99）
句子 6 是因果句（置信度: 0.98）
句子 7 是因果句（置信度: 0.99）
句子 8 是因果句（置信度: 0.98）
句子 9 是因果句（置信度: 0.99）
句子 10 是因果句（置信度: 0.99）
句子 11 是因果句（置信度: 0.99）
句子 12 是因果句（置信度: 0.99）
句子 13 不是因果句（置信度: 0.01）
句子 14 不是因果句（置信度: 0.05）

共检测到 12 个因果句


## 3. 因果跨度提取

使用专门训练的**中文 BERT 模型**提取原因和结果部分：
- 模型：`causal-narrative/zh-roberta-causal-span-extractor`
- 基于中文 RoBERTa，在中文因果跨度标注数据上微调

In [4]:
import pandas as pd
from causal_narrative.extraction import CausalSpanExtractor

# 使用中文 BERT 模型进行跨度提取
logger.info("初始化因果跨度提取器（中文 BERT 模型）...")
extractor = CausalSpanExtractor(
    method='bert',
    bert_model_name='causal-narrative/zh-roberta-causal-span-extractor'  # 使用中文模型
)

logger.info("提取原因-结果跨度...")
span_results = extractor.extract(causal_sentences, show_progress=True)

valid_spans = []
for i, span in enumerate(span_results):
    if span and span.cause_text and span.effect_text:
        valid_spans.append({
            'sentence': causal_sentences[i],
            'cause_text': span.cause_text,
            'effect_text': span.effect_text
        })
        print(f"\n句子: {causal_sentences[i]}")
        print(f"  原因:  {span.cause_text}")
        print(f"  结果: {span.effect_text}")
    else:
        print(f"\n句子: {causal_sentences[i]} - 未提取到跨度")

df_spans = pd.DataFrame(valid_spans)
print(f"\n提取了 {len(df_spans)} 个有效的因果跨度。")

2026-03-25 17:14:02.646 | INFO     | __main__:<module>:5 - 初始化因果跨度提取器（中文 BERT 模型）...
2026-03-25 17:14:02.647 | INFO     | causal_narrative.extraction:_resolve_bert_model_source:544 - Loading BERT model from local cache: model/causal-narrative_zh-roberta-causal-span-extractor
2026-03-25 17:14:02.733 | INFO     | causal_narrative.extraction:__init__:636 - BERTSpanExtractor loaded from: model/causal-narrative_zh-roberta-causal-span-extractor
2026-03-25 17:14:02.733 | INFO     | causal_narrative.extraction:__init__:637 - Device: cpu
2026-03-25 17:14:02.733 | INFO     | causal_narrative.extraction:__init__:988 - CausalSpanExtractor initialized with BERT method (model=causal-narrative/zh-roberta-causal-span-extractor, cache_dir=model)
2026-03-25 17:14:02.734 | INFO     | __main__:<module>:11 - 提取原因-结果跨度...
2026-03-25 17:14:02.734 | INFO     | causal_narrative.extraction:_extract_bert_batch:1218 - Starting batch BERT extraction for 12 sentences
BERT extraction: 100%|██████████| 12/12 [00:00<0


句子: 央行提高了利率，导致物价下降。
  原因:  央行提高了利率
  结果: 物价下降

句子: 中央银行上调了基准利率，导致经济严重通缩。
  原因:  银行上调了基准利率
  结果: 经济严重通缩

句子: 企业大幅裁员，导致失业率攀升
  原因:  企业大幅裁员
  结果: 失业率攀升

句子: 央行提高了利率，导致金融系统崩溃。
  原因:  央行提高了利率
  结果: 金融系统崩溃

句子: 中央银行上调了基准利率，导致经济严重萎缩。
  原因:  银行上调了基准利率
  结果: 经济严重萎缩

句子: 贸易战加剧，导致出口锐减。
  原因:  贸易战加剧
  结果: 出口锐减

句子: 政府削减了公共支出，导致经济衰退。
  原因:  政
  结果: 经济衰退

句子: 政府减少了预算，导致经济增长停滞。
  原因:  政府减少了预算
  结果: 经济增长停滞

句子: 国家大幅削减了资金，导致经济陷入低迷。
  原因:  大幅削减了资金
  结果: 经济陷入低迷

句子: 疫情爆发，导致经济停滞。
  原因:  疫情爆发
  结果: 经济停滞

句子: 政府减少了预算，导致公共服务瘫痪。
  原因:  政府
  结果: 公共服务瘫痪

句子: 国家大幅削减了资金，导致医疗系统崩溃。
  原因:  大幅削减了资金
  结果: 医疗系统崩溃

提取了 12 个有效的因果跨度。


## 4. 语义角色标注（使用 HanLP）

**重要**：需要先安装 HanLP：`pip install hanlp`

首次运行时，HanLP 会下载模型，可能需要一些时间。

In [5]:
# 由于 HanLP 与 transformers 版本不兼容，使用 jieba 作为中文 SRL 备用方案
import jieba
import jieba.posseg as pseg

def chinese_srl_jieba(text):
    """使用 jieba 进行简化的中文 SRL"""
    if not text or len(text.strip()) < 2:
        return {'words': [], 'verbs': []}
    
    # 使用 jieba 进行词性标注
    words_pos = list(pseg.cut(text))
    words = [w for w, p in words_pos]
    
    # 寻找动词
    verb = None
    verb_idx = -1
    for idx, (word, pos) in enumerate(words_pos):
        # v 开头的词性是动词
        if pos.startswith('v'):
            verb = word
            verb_idx = idx
            break
    
    if not verb:
        return {'words': words, 'verbs': []}
    
    # 简单规则：动词前是 ARG0（施事），动词后是 ARG1（受事）
    arg0_words = words[:verb_idx] if verb_idx > 0 else []
    arg1_words = words[verb_idx+1:] if verb_idx < len(words)-1 else []
    
    arg0 = ''.join(arg0_words) if arg0_words else None
    arg1 = ''.join(arg1_words) if arg1_words else None
    
    # 构建描述
    description_parts = []
    if arg0:
        description_parts.append(f"[ARG0: {arg0}]")
    description_parts.append(f"[V: {verb}]")
    if arg1:
        description_parts.append(f"[ARG1: {arg1}]")
    description = " ".join(description_parts)
    
    # 构建 tags
    tags = ["O"] * len(words)
    if 0 <= verb_idx < len(tags):
        tags[verb_idx] = "B-V"
    
    return {
        'words': words,
        'verbs': [{
            'verb': verb,
            'description': description,
            'tags': tags
        }]
    }

logger.info("使用 jieba 对原因和结果跨度运行 SRL...")
cause_srl_results = [chinese_srl_jieba(text) for text in df_spans['cause_text'].tolist()]
effect_srl_results = [chinese_srl_jieba(text) for text in df_spans['effect_text'].tolist()]

df_spans['cause_srl'] = cause_srl_results
df_spans['effect_srl'] = effect_srl_results

print("\n--- SRL 结果示例 ---")
for idx, row in df_spans.head(3).iterrows():
    print(f"\n句子: {row['sentence']}")
    print(f"  原因跨度: {row['cause_text']}")
    print(f"  原因 SRL: {row['cause_srl']}")
    if row['cause_srl'].get('verbs'):
        for verb in row['cause_srl']['verbs']:
            print(f"    → {verb['description']}")
    print(f"  结果跨度: {row['effect_text']}")
    print(f"  结果 SRL: {row['effect_srl']}")
    if row['effect_srl'].get('verbs'):
        for verb in row['effect_srl']['verbs']:
            print(f"    → {verb['description']}")

2026-03-25 17:14:05.882 | INFO     | __main__:<module>:57 - 使用 jieba 对原因和结果跨度运行 SRL...
Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/xx/0g3688s12k37v7m72x41bp400000gn/T/jieba.cache
Loading model cost 0.245 seconds.
Prefix dict has been built successfully.



--- SRL 结果示例 ---

句子: 央行提高了利率，导致物价下降。
  原因跨度: 央行提高了利率
  原因 SRL: {'words': ['央行', '提高', '了', '利率'], 'verbs': [{'verb': '提高', 'description': '[ARG0: 央行] [V: 提高] [ARG1: 了利率]', 'tags': ['O', 'B-V', 'O', 'O']}]}
    → [ARG0: 央行] [V: 提高] [ARG1: 了利率]
  结果跨度: 物价下降
  结果 SRL: {'words': ['物价', '下降'], 'verbs': [{'verb': '下降', 'description': '[ARG0: 物价] [V: 下降]', 'tags': ['O', 'B-V']}]}
    → [ARG0: 物价] [V: 下降]

句子: 中央银行上调了基准利率，导致经济严重通缩。
  原因跨度: 银行上调了基准利率
  原因 SRL: {'words': ['银行', '上调', '了', '基准利率'], 'verbs': [{'verb': '上调', 'description': '[ARG0: 银行] [V: 上调] [ARG1: 了基准利率]', 'tags': ['O', 'B-V', 'O', 'O']}]}
    → [ARG0: 银行] [V: 上调] [ARG1: 了基准利率]
  结果跨度: 经济严重通缩
  结果 SRL: {'words': ['经济', '严重', '通缩'], 'verbs': []}

句子: 企业大幅裁员，导致失业率攀升
  原因跨度: 企业大幅裁员
  原因 SRL: {'words': ['企业', '大幅', '裁员'], 'verbs': []}
  结果跨度: 失业率攀升
  结果 SRL: {'words': ['失业率', '攀升'], 'verbs': [{'verb': '攀升', 'description': '[ARG0: 失业率] [V: 攀升]', 'tags': ['O', 'B-V']}]}
    → [ARG0: 失业率] [V: 攀升]


## 5. 事件聚类（使用中文 BERT 模型）

使用中文 BERT embedding 模型进行聚类：

In [6]:
from causal_narrative.embedding import (
    SentenceEmbedder,
    generate_role_based_embeddings,
    generate_phrase_embeddings,
    DEFAULT_CHINESE_MODEL_NAME
)
from causal_narrative.event_clustering import (
    run_hdbscan,
    generate_cluster_names_from_srl,
    generate_cluster_names_from_texts
)
from causal_narrative.semantic_role_labeling import is_event_srl
import numpy as np

# 初始化中文 embedding 模型
print(f"\n初始化中文 embedding 模型: {DEFAULT_CHINESE_MODEL_NAME}")
embedder = SentenceEmbedder(model_name=DEFAULT_CHINESE_MODEL_NAME)

# 检查 SRL 有效性
df_spans['cause_valid_for_role'] = df_spans['cause_srl'].apply(is_event_srl)
df_spans['effect_valid_for_role'] = df_spans['effect_srl'].apply(is_event_srl)

print(f"原因可用于角色聚类: {df_spans['cause_valid_for_role'].sum()}/{len(df_spans)}")
print(f"结果可用于角色聚类: {df_spans['effect_valid_for_role'].sum()}/{len(df_spans)}")

# 初始化聚类列
df_spans['cause_cluster_id'] = -1
df_spans['cause_cluster_name'] = ''
df_spans['effect_cluster_id'] = -1
df_spans['effect_cluster_name'] = ''


初始化中文 embedding 模型: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


2026-03-25 17:14:14.525 | INFO     | causal_narrative.embedding:__init__:91 - Loaded embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 (dimension: 384)


原因可用于角色聚类: 9/12
结果可用于角色聚类: 10/12


In [7]:
# --- 原因聚类 ---
print("\n--- 原因聚类 ---")

# 1. 基于角色的聚类
cause_role_mask = df_spans['cause_valid_for_role']
if cause_role_mask.any():
    print(f"聚类 {cause_role_mask.sum()} 个基于角色的原因...")
    cause_srl_list = df_spans.loc[cause_role_mask, 'cause_srl'].tolist()
    
    embeddings_cause_role = generate_role_based_embeddings(
        srl_results=cause_srl_list,
        embedder=embedder,
        batch_size=32,
        show_progress=False
    )
    
    c_ids, _ = run_hdbscan(embeddings_cause_role, min_cluster_size=2)
    
    names = generate_cluster_names_from_srl(
        labels=c_ids,
        srl_results=cause_srl_list,
        fallback_texts=df_spans.loc[cause_role_mask, 'cause_text'].tolist()
    )
    
    df_spans.loc[cause_role_mask, 'cause_cluster_id'] = c_ids
    df_spans.loc[cause_role_mask, 'cause_cluster_name'] = [names[cid] for cid in c_ids]

# 2. 基于短语的聚类
cause_phrase_mask = ~cause_role_mask
if cause_phrase_mask.any():
    print(f"聚类 {cause_phrase_mask.sum()} 个基于短语的原因...")
    cause_texts = df_spans.loc[cause_phrase_mask, 'cause_text'].tolist()
    
    embeddings_cause_phrase = generate_phrase_embeddings(
        texts=cause_texts,
        embedder=embedder,
        batch_size=32,
        show_progress=False
    )
    
    c_ids, _ = run_hdbscan(embeddings_cause_phrase, min_cluster_size=2)
    
    names = generate_cluster_names_from_texts(
        labels=c_ids,
        texts=cause_texts
    )
    
    max_id = df_spans['cause_cluster_id'].max()
    offset = max_id + 2
    c_ids_shifted = c_ids + offset
    
    df_spans.loc[cause_phrase_mask, 'cause_cluster_id'] = c_ids_shifted
    df_spans.loc[cause_phrase_mask, 'cause_cluster_name'] = [names[cid] for cid in c_ids]

2026-03-25 17:14:17.279 | INFO     | causal_narrative.embedding:generate_role_based_embeddings:401 - Generating role-based embeddings for 9 SRL results



--- 原因聚类 ---
聚类 9 个基于角色的原因...


2026-03-25 17:14:17.659 | INFO     | causal_narrative.embedding:generate_role_based_embeddings:461 - Role-based embeddings generated: shape=(9, 1152)
2026-03-25 17:14:17.872 | INFO     | causal_narrative.event_clustering:run_hdbscan:441 - Running HDBSCAN: min_cluster_size=2, min_samples=2, metric=euclidean, n_samples=9
2026-03-25 17:14:17.875 | INFO     | causal_narrative.event_clustering:run_hdbscan:481 - HDBSCAN completed: 2 clusters, 2 noise points (22.2%)
2026-03-25 17:14:17.876 | INFO     | causal_narrative.embedding:generate_phrase_embeddings:490 - Generating phrase embeddings for 3 texts
2026-03-25 17:14:18.059 | INFO     | causal_narrative.embedding:generate_phrase_embeddings:508 - Phrase embeddings generated: shape=(3, 384)
2026-03-25 17:14:18.059 | INFO     | causal_narrative.event_clustering:run_hdbscan:441 - Running HDBSCAN: min_cluster_size=2, min_samples=2, metric=euclidean, n_samples=3
2026-03-25 17:14:18.060 | INFO     | causal_narrative.event_clustering:run_hdbscan:481

聚类 3 个基于短语的原因...


In [8]:
# --- 结果聚类 ---
print("\n--- 结果聚类 ---")

max_cause_id = df_spans['cause_cluster_id'].max()
effect_id_offset = max_cause_id + 100
print(f"对结果聚类应用偏移量 {effect_id_offset} 以避免 ID 冲突")

# 1. 基于角色的聚类
effect_role_mask = df_spans['effect_valid_for_role']
if effect_role_mask.any():
    print(f"聚类 {effect_role_mask.sum()} 个基于角色的结果...")
    effect_srl_list = df_spans.loc[effect_role_mask, 'effect_srl'].tolist()
    
    embeddings_effect_role = generate_role_based_embeddings(
        srl_results=effect_srl_list,
        embedder=embedder,
        batch_size=32,
        show_progress=False
    )
    
    c_ids, _ = run_hdbscan(embeddings_effect_role, min_cluster_size=2)
    
    names = generate_cluster_names_from_srl(
        labels=c_ids,
        srl_results=effect_srl_list,
        fallback_texts=df_spans.loc[effect_role_mask, 'effect_text'].tolist()
    )
    
    c_ids_shifted = c_ids + effect_id_offset
    
    df_spans.loc[effect_role_mask, 'effect_cluster_id'] = c_ids_shifted
    df_spans.loc[effect_role_mask, 'effect_cluster_name'] = [names[cid] for cid in c_ids]

# 2. 基于短语的聚类
effect_phrase_mask = ~effect_role_mask
if effect_phrase_mask.any():
    print(f"聚类 {effect_phrase_mask.sum()} 个基于短语的结果...")
    effect_texts = df_spans.loc[effect_phrase_mask, 'effect_text'].tolist()
    
    embeddings_effect_phrase = generate_phrase_embeddings(
        texts=effect_texts,
        embedder=embedder,
        batch_size=32,
        show_progress=False
    )
    
    c_ids, _ = run_hdbscan(embeddings_effect_phrase, min_cluster_size=2)
    
    names = generate_cluster_names_from_texts(
        labels=c_ids,
        texts=effect_texts
    )
    
    max_id = df_spans['effect_cluster_id'].max()
    offset = max_id + 2
    c_ids_shifted = c_ids + offset
    
    df_spans.loc[effect_phrase_mask, 'effect_cluster_id'] = c_ids_shifted
    df_spans.loc[effect_phrase_mask, 'effect_cluster_name'] = [names[cid] for cid in c_ids]

2026-03-25 17:14:20.619 | INFO     | causal_narrative.embedding:generate_role_based_embeddings:401 - Generating role-based embeddings for 10 SRL results
2026-03-25 17:14:20.758 | INFO     | causal_narrative.embedding:generate_role_based_embeddings:461 - Role-based embeddings generated: shape=(10, 1152)
2026-03-25 17:14:20.759 | INFO     | causal_narrative.event_clustering:run_hdbscan:441 - Running HDBSCAN: min_cluster_size=2, min_samples=2, metric=euclidean, n_samples=10
2026-03-25 17:14:20.759 | INFO     | causal_narrative.event_clustering:run_hdbscan:481 - HDBSCAN completed: 0 clusters, 10 noise points (100.0%)
2026-03-25 17:14:20.761 | INFO     | causal_narrative.embedding:generate_phrase_embeddings:490 - Generating phrase embeddings for 2 texts



--- 结果聚类 ---
对结果聚类应用偏移量 102 以避免 ID 冲突
聚类 10 个基于角色的结果...
聚类 2 个基于短语的结果...


2026-03-25 17:14:21.076 | INFO     | causal_narrative.embedding:generate_phrase_embeddings:508 - Phrase embeddings generated: shape=(2, 384)
2026-03-25 17:14:21.077 | INFO     | causal_narrative.event_clustering:run_hdbscan:441 - Running HDBSCAN: min_cluster_size=2, min_samples=2, metric=euclidean, n_samples=2
2026-03-25 17:14:21.077 | INFO     | causal_narrative.event_clustering:run_hdbscan:481 - HDBSCAN completed: 0 clusters, 2 noise points (100.0%)


In [9]:
print("\n--- 聚类结果 ---")
print("原因聚类:")
print(df_spans[['cause_text', 'cause_cluster_id', 'cause_cluster_name']].sort_values('cause_cluster_id').to_string())
print("\n结果聚类:")
print(df_spans[['effect_text', 'effect_cluster_id', 'effect_cluster_name']].sort_values('effect_cluster_id').to_string())


--- 聚类结果 ---
原因聚类:
   cause_text  cause_cluster_id cause_cluster_name
5       贸易战加剧                -1             贸易战 加剧
9        疫情爆发                -1             贸易战 加剧
0     央行提高了利率                 0          央行 提高 了利率
1   银行上调了基准利率                 0          央行 提高 了利率
3     央行提高了利率                 0          央行 提高 了利率
4   银行上调了基准利率                 0          央行 提高 了利率
7     政府减少了预算                 1          大幅 削减 了资金
8     大幅削减了资金                 1          大幅 削减 了资金
11    大幅削减了资金                 1          大幅 削减 了资金
2      企业大幅裁员                 2             企业大幅裁员
6           政                 2             企业大幅裁员
10         政府                 2             企业大幅裁员

结果聚类:
   effect_text  effect_cluster_id effect_cluster_name
0         物价下降                101               物价 下降
2        失业率攀升                101               物价 下降
3       金融系统崩溃                101               物价 下降
4       经济严重萎缩                101               物价 下降
5         出口锐减                101       

## 6. 因果网络可视化

In [10]:
from causal_narrative.network import CausalNetworkBuilder
from causal_narrative.viz import visualize_causal_network

# 初始化构建器
builder = CausalNetworkBuilder()

# 从 DataFrame 构建网络
builder.build_from_dataframe(
    df_spans,
    cause_col='cause_cluster_id',
    effect_col='effect_cluster_id',
    cause_text_col='cause_cluster_name',
    effect_text_col='effect_cluster_name'
)

# 显式设置节点标签
G = builder.graph
for node in G.nodes():
    # 尝试从原因聚类中找到名称
    cause_match = df_spans[df_spans['cause_cluster_id'] == node]
    if not cause_match.empty:
        label = cause_match.iloc[0]['cause_cluster_name']
        G.nodes[node]['label'] = label
        continue
    
    # 尝试从结果聚类中找到名称
    effect_match = df_spans[df_spans['effect_cluster_id'] == node]
    if not effect_match.empty:
        label = effect_match.iloc[0]['effect_cluster_name']
        G.nodes[node]['label'] = label
        continue
    
    # 后备方案
    G.nodes[node]['label'] = f"聚类 {node}"

# 可视化网络
visualize_causal_network(builder.graph, output_html='causal_network_zh.html')
print("因果网络已保存到 causal_network_zh.html")

2026-03-25 17:14:33.866 | INFO     | causal_narrative.network:build_from_dataframe:108 - Building network from DataFrame with 12 causal pairs...
2026-03-25 17:14:33.868 | INFO     | causal_narrative.network:build_from_dataframe:123 - Network built successfully: 6 nodes, 6 edges



Generating interactive network...
  Nodes: 6
  Edges: 6
  ✓ HTML saved: causal_network_zh.html
因果网络已保存到 causal_network_zh.html
